In [113]:
import os
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset
import numpy as np


class SequenceDataset(Dataset):
    def __init__(self, file_path, label_path, split):
        file = torch.load(file_path, map_location=torch.device('cpu'))
        # Extract the lists from the dictionary
        ids = file["ids"]               # List of strings
        # DeepPL files contains accession id without version number, so we need to erase it from our ids
        ids = [item.split('.')[0] for item in ids] 
        embeddings_list = file["embeddings"] # List of torch.Tensors
        embeddings_tensor = torch.stack(embeddings_list)
        # To Numpy for Pandas
        loaded_ids = np.array(ids)
        loaded_embeds = embeddings_tensor.detach().cpu().numpy()

        data = pd.DataFrame(data=loaded_embeds[:,:], index=loaded_ids[:], columns=[[f"feature_{i}" for i in range(len(loaded_embeds[0]))]]) 
        data = (data-data.min())/(data.max()-data.min())


        # Label
        # --- CONFIGURATION ---
        # Training
        sheet_names = pd.ExcelFile(label_path).sheet_names
        all_metadata = pd.DataFrame()
        # Training dataset
        if split.strip().lower() == 'train':
            all_metadata = pd.read_excel(label_path, sheet_name=sheet_names[0])
        # Test dataset
        if split.strip().lower() == 'test':
            all_metadata = pd.read_excel(label_path, sheet_name=sheet_names[1])

        lifecycle_map = {}
        # all_metadata = pd.concat([training, test])

        for _, row in all_metadata.iterrows():
            acc = row['Accession number']
            raw_life = row['Lifecycle']
            clean_life = "lysogenic" if str(raw_life).strip().lower() in ["lysogneic", "lysogenic"] else "lytic"
            lifecycle_map[acc] = clean_life

        # Create aligned list
        aligned_lifecycles = []
        # self.data = pd.DataFrame(columns=['accession', [f'feature_{i}' for i in range(len(loaded_embeds[0]))], 'lifecycle'])
        self.data = pd.DataFrame(columns=['accession', 'embedding', 'lifecycle'])
        for accession in data.index:
            if accession in lifecycle_map:
                #aligned_lifecycles.append(lifecycle_map[accession])
                # LABEL: 0 = lysogenic, 1 = lythic
                self.data.loc[len(self.data)] = [accession, data.loc[accession], 0 if lifecycle_map[accession] == 'lysogenic' else 1]
            # else:
            #     # Warning only once per missing ID to avoid clutter
            #     print(f"Warning: Accession {accession} not found. Defaulting to lytic.")
            #     # aligned_lifecycles.append("lytic") 
            #     self.data.loc[len(self.data)] = [accession, all_metadata.loc[accession], 'lytic']
        
        self.labels = aligned_lifecycles

    def __len__(self):
        return len(self.data.index)

    def __getitem__(self, idx):
        # return self.data.iloc[idx].to_numpy(), self.labels[idx]
        return self.data.iloc[idx]['accession'], torch.from_numpy(self.data.iloc[idx]['embedding'].to_numpy()), self.data.iloc[idx]['lifecycle']

In [114]:
file_path = 'embeddings_171225_2176_32768_blocks.28.mlp.l3' 
label_path = "journal.pcbi.1012525.s001.xlsx"


training_data = SequenceDataset(file_path, label_path, 'train')
test_data = SequenceDataset(file_path, label_path, 'test')


train_dataloader = DataLoader(training_data, batch_size=64, shuffle=True)
test_dataloader = DataLoader(test_data, batch_size=64, shuffle=True)

#print(training_data[0][1])
print(next(iter(train_dataloader)))

[('NC_022791', 'NC_008376', 'NC_029033', 'NC_041845', 'NC_019506', 'NC_042044', 'NC_030476', 'NC_042096', 'NC_018850', 'NC_027298', 'NC_020843', 'NC_009763', 'NC_002643', 'NC_006882', 'NC_011703', 'NC_041926', 'NC_028760', 'NC_042326', 'NC_008717', 'NC_019914', 'NC_021852', 'NC_028835', 'NC_023743', 'NC_027355', 'NC_028685', 'NC_041890', 'NC_016766', 'NC_013155', 'NC_041910', 'NC_019450', 'NC_010179', 'NC_019457', 'NC_041952', 'NC_024389', 'NC_021775', 'AM084414', 'NC_028850', 'NC_042008', 'NC_004589', 'NC_041995', 'NC_041987', 'NC_041877', 'NC_015296', 'NC_041912', 'NC_022764', 'NC_021539', 'NC_041990', 'NC_018280', 'NC_042342', 'NC_024388', 'NC_024786', 'NC_021331', 'NC_023575', 'NC_018085', 'NC_027398', 'NC_011085', 'NC_010237', 'NC_021561', 'NC_029003', 'NC_020480', 'NC_025462', 'NC_042010', 'NC_019447', 'NC_042354'), tensor([[0.0478, 0.9651, 0.8832,  ..., 0.9392, 0.6473, 0.6858],
        [0.0642, 0.9266, 0.8645,  ..., 0.9345, 0.6507, 0.6445],
        [0.0600, 0.9273, 0.8373,  ...,

In [115]:
import torch.nn as nn
import torch.optim as optim

class PhageLifecycleMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim=8192):
        super(PhageLifecycleMLP, self).__init__()
        self.network = nn.Sequential(
            # Input Layer -> Hidden Layer 1
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.4), # Helps prevent overfitting
            
            # Hidden Layer 1 -> Hidden Layer 2
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(0.2),

            # Hidden Layer 2 -> Hidden Layer 3
            nn.Linear(hidden_dim // 2, hidden_dim // 4),
            nn.ReLU(),
            
            # Hidden Layer 2 -> Output Layer (1 unit for binary classification)
            nn.Linear(hidden_dim // 4, 1) 
        )

    def forward(self, x):
        return self.network(x)
    
# --- 1. Detect Input Dimension Automatically ---
# We grab one batch to see how many features are in your embeddings
first_batch = next(iter(train_dataloader))
_, sample_inputs, _ = first_batch
input_dim = sample_inputs.shape[1] 

print(f"Detected Input Feature Size: {input_dim}")
print(f"Detected Batch Size: {sample_inputs.shape[0]}")

# --- 2. Initialize Model ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = PhageLifecycleMLP(input_dim=input_dim).to(device)

# Loss and Optimizer
# BCEWithLogitsLoss is numerically stable for binary classification
criterion = nn.BCEWithLogitsLoss() 
optimizer = optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-5)

# --- 3. Training Loop ---
num_epochs = 1000

print("\nStarting training...")
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    # UNPACKING: We explicitly ignore the first element (IDs) using '_'
    for batch_idx, (_, inputs, labels) in enumerate(train_dataloader):
        
        # Move inputs to GPU/CPU and ensure float32
        inputs = inputs.float().to(device)
        
        # Reshape labels to (Batch_Size, 1) and ensure float32 for BCE Loss
        labels = labels.float().unsqueeze(1).to(device)
        
        # Zero gradients
        optimizer.zero_grad()
        
        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        # Metrics
        running_loss += loss.item()
        
        # Apply sigmoid to convert logits to probabilities -> round to 0 or 1
        predicted = (torch.sigmoid(outputs) > 0.5).float()
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_acc = 100 * correct / total
    print(f"Epoch [{epoch+1}/{num_epochs}] | Loss: {running_loss/len(train_dataloader):.4f} | Accuracy: {epoch_acc:.2f}%")

print("Training finished.")

Detected Input Feature Size: 4096
Detected Batch Size: 64

Starting training...
Epoch [1/1000] | Loss: 0.9357 | Accuracy: 65.13%
Epoch [2/1000] | Loss: 0.5608 | Accuracy: 66.30%
Epoch [3/1000] | Loss: 0.5500 | Accuracy: 67.24%
Epoch [4/1000] | Loss: 0.5421 | Accuracy: 67.02%
Epoch [5/1000] | Loss: 0.5404 | Accuracy: 68.63%
Epoch [6/1000] | Loss: 0.5398 | Accuracy: 66.74%
Epoch [7/1000] | Loss: 0.5328 | Accuracy: 68.46%
Epoch [8/1000] | Loss: 0.5291 | Accuracy: 70.01%
Epoch [9/1000] | Loss: 0.5389 | Accuracy: 68.74%
Epoch [10/1000] | Loss: 0.5301 | Accuracy: 70.73%
Epoch [11/1000] | Loss: 0.5075 | Accuracy: 72.89%
Epoch [12/1000] | Loss: 0.4994 | Accuracy: 73.67%
Epoch [13/1000] | Loss: 0.4894 | Accuracy: 75.00%
Epoch [14/1000] | Loss: 0.4882 | Accuracy: 75.67%
Epoch [15/1000] | Loss: 0.4573 | Accuracy: 78.94%
Epoch [16/1000] | Loss: 0.4461 | Accuracy: 79.38%
Epoch [17/1000] | Loss: 0.4477 | Accuracy: 79.71%
Epoch [18/1000] | Loss: 0.4561 | Accuracy: 78.10%
Epoch [19/1000] | Loss: 0.436

KeyboardInterrupt: 

In [116]:
model.eval()
test_correct = 0
test_total = 0

print("\nEvaluating on Test Set...")
with torch.no_grad():
    for _, inputs, labels in test_dataloader:
        inputs = inputs.float().to(device)
        labels = labels.float().unsqueeze(1).to(device)
        
        outputs = model(inputs)
        predicted = (torch.sigmoid(outputs) > 0.5).float()
        
        test_total += labels.size(0)
        test_correct += (predicted == labels).sum().item()

print(f"Test Accuracy: {100 * test_correct / test_total:.2f}%")


Evaluating on Test Set...
Test Accuracy: 84.45%
